# 2장 실습: 네이버 지식인 크롤링 🍲

requests로 HTML을 받아오고, BeautifulSoup으로 질문의 **제목, 링크, 날짜, 카테고리, 답변 수**를 뽑아서 파일로 저장합니다.

| Step | 할 일 |
|---|---|
| 0 | HTML 받아오기 (requests) |
| 1 | 질문 **1개** 뽑기 (select_one) |
| 2 | 한 페이지 질문 **전부** 뽑기 (select + for문) |
| 3 | **여러 페이지** 뽑기 |
| 4 | DataFrame으로 정리 |
| 5 | CSV, JSON, 엑셀로 저장 |

> ✏️ `____` 로 된 빈칸을 채우면서 따라오세요. 막히면 노션 강의자료를 참고하세요!

## 준비: 라이브러리 불러오기

In [ ]:
import time

import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://kin.naver.com/search/list.naver"

# User-Agent: "저 로봇 아니고 크롬 브라우저예요" 하는 신분증. 없으면 막는 사이트가 많아요.
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0 Safari/537.36"}

ModuleNotFoundError: No module named 'request'

## Step 0. HTML 받아오기

`requests.get()`으로 설명서(HTML)를 주문합니다. 한글 검색어는 `params`에 넣으면 requests가 알아서 인코딩해줘요.

In [ ]:
response = requests.get(URL, params={"query": "삼성전자"}, headers=HEADERS, timeout=10)

print(response.status_code)   # 200이면 성공! (404: 없는 주소, 403: 거절당함)
html = response.text          # 도착한 설명서(HTML) 문자열
print(html[:500])

> 🚨 **네이버 접속이 막혔다면?** 아래 셀의 주석(`#`)을 풀고 실행해서 백업 파일로 진행하세요. (Step 3은 백업으로는 1페이지만 됩니다)

In [ ]:
# with open("kin_backup.html", encoding="utf-8") as f:
#     html = f.read()

## Step 1. 질문 1개 뽑기

1장에서 본 구조 기억나시죠? 질문 하나는 `ul.basic1` 안의 `li` 안의 `dl` 한 덩어리입니다.

- `select_one("선택자")`: 조건에 맞는 **첫 번째 하나**만 찾기

In [ ]:
soup = BeautifulSoup(html, "html.parser")   # 설명서를 BeautifulSoup으로 펼치기

item = soup.select_one(".basic1 > li > dl")   # 첫 번째 질문 덩어리
print(item.prettify()[:1000])

이제 덩어리 안에서 필요한 조각을 꺼냅니다.

- 제목 `a` 태그의 class: `_nclicks:qna.txt` → 선택자에서 `:`와 `.`는 앞에 `\\`를 붙여야 해요
- `.text`: 상자 안의 **내용물** / `["href"]`: 송장에 적힌 **링크 주소**

In [6]:
title_tag = item.select_one("____")

title = title_tag.____.strip()   # strip(): 앞뒤 공백 제거
link = title_tag["____"]
date = item.select_one("____").text.strip()
category = item.select_one("._nclicks\\:qna\\.cat2").text.strip()
answers = item.select_one(".hit").text.split()[-1]   # "답변수 3" -> ["답변수", "3"] -> "3"

print(title, link, date, category, answers, sep=" | ")

NameError: name 'item' is not defined

## Step 2. 한 페이지 질문 전부 뽑기

- `select("선택자")`: 조건에 맞는 것을 **전부** 찾아서 리스트로

찾은 질문 덩어리들을 for문으로 돌면서 Step 1을 반복하면 끝!

In [ ]:
items = soup.select(".basic1 > li > dl")
print(len(items), "개 찾음")

rows = []
for item in items:
    title_tag = item.select_one("._nclicks\\:qna\\.txt")
    rows.append({
        "title": title_tag.text.strip(),
        "link": title_tag["href"],
        "date": item.select_one(".txt_inline").text.strip(),
        "category": item.select_one("._nclicks\\:qna\\.cat2").text.strip(),
        "answers": item.select_one(".hit").text.split()[-1],
    })

rows[:3]

## Step 3. 여러 페이지 뽑기

주소 끝의 `page=` 숫자만 바꾸면 다음 페이지입니다. 페이지마다 요청 → 파싱 → 추출을 반복해요.

> ⚠️ 서버에 한꺼번에 요청을 몰아 보내면 차단당할 수 있어요. `time.sleep()`으로 페이지 사이에 잠깐 쉬어줍니다.

In [ ]:
rows = []
for page in range(1, 4):   # 1, 2, 3페이지
    response = requests.get(URL, params={"query": "삼성전자", "page": ____}, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    for item in soup.select(".basic1 > li > dl"):
        title_tag = item.select_one("._nclicks\\:qna\\.txt")
        rows.append({
            "title": title_tag.text.strip(),
            "link": title_tag["href"],
            "date": item.select_one(".txt_inline").text.strip(),
            "category": item.select_one("._nclicks\\:qna\\.cat2").text.strip(),
            "answers": item.select_one(".hit").text.split()[-1],
        })

    print(f"{page}페이지 완료, 지금까지 {len(rows)}개")
    time.sleep(0.5)

## Step 4. DataFrame으로 정리하기

딕셔너리 리스트를 pandas에 넣으면 바로 표가 됩니다.

In [ ]:
df = pd.DataFrame(rows)
df.head()

## Step 5. 파일로 저장하기

| 형식 | 특징 |
|---|---|
| CSV | 엑셀로 바로 열림. 가볍고 단순 |
| JSON | 웹, API에서 주로 쓰는 형식. 중첩 구조 가능 |
| 엑셀 | 그대로 공유하기 편함 |

> 💡 `encoding="utf-8-sig"`: 윈도우 엑셀에서 CSV를 열 때 **한글이 깨지지 않게** 해줍니다.

In [ ]:
df.to_csv("kin.csv", index=False, encoding="utf-8-sig")
df.to_json("kin.json", orient="records", force_ascii=False, indent=2)
df.to_excel("kin.xlsx", index=False)
print("저장 완료!")

## 🎉 끝! 그리고 4장 예고

지금까지 짠 코드를 함수 하나로 정리하면 이렇게 됩니다.

```python
def crawl_kin(query: str, pages: int = 1) -> list[dict]:
    ...  # Step 0~3을 그대로
    return rows
```

4장에서는 이 함수에 딱 한 줄을 붙여서 **AI가 직접 호출할 수 있는 도구**로 만들어봅니다. 🔌